In [1]:
import pandas as pd

In [2]:
from colbert.evaluation.evaluator import ColBERTEvaluator

In [3]:
from colbert.infra import ColBERTConfig

In [4]:
config = ColBERTConfig(bsize=32, lr=1e-05, warmup=20_000, doc_maxlen=512, dim=128, attend_to_mask_tokens=False, nway=64, accumsteps=1, similarity='cosine', use_ib_negatives=True)

In [5]:
evaluator=ColBERTEvaluator(config=config,checkpoint_path='bert-base-uncased')

Some weights of HF_ColBERT were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['linear.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[Mar 06, 06:54:40] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


In [6]:
import os
eval_out_path='/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/eval_out'
os.makedirs(eval_out_path,exist_ok=True)

In [7]:
from pathlib import Path
p=Path('baleen')

In [8]:
from colbert.evaluation.triplet_loader import convert_triplets_to_qrels

In [9]:
queries_path='/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/queries.train.colbert.tsv'
triplets_path='/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/triples.train.colbert.jsonl'
qrels_path = convert_triplets_to_qrels(triplets_path,output_path=os.path.join(Path(queries_path).parent,'qrels.train.colbert.tsv')) if triplets_path else None

#> Loading triplets from /Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/triples.train.colbert.jsonl
#> Loaded 9 queries with positives
#> Average positives per query: 1.56
#> Average negatives per query: 1.33
#> Converted triplets to qrels format: /Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/qrels.train.colbert.tsv


In [10]:
p.root

''

In [11]:
eval_res=evaluator.evaluate(
        queries_path='/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/queries.train.colbert.tsv',
        collection_path='/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/corpus.train.colbert.tsv',
        triplets_path='/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/triples.train.colbert.jsonl',
        output_path=os.path.join(eval_out_path,'output.train.colbert.tsv'),
        batch_size=128,
        depth=1000,
        step=None
    )

[Mar 06, 06:54:40] #> Loading the queries from /Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/queries.train.colbert.tsv ...
[Mar 06, 06:54:40] #> Got 20 queries. All QIDs are unique.

[Mar 06, 06:54:40] #> Loading collection...
0M 
#> Loading triplets from /Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/triples.train.colbert.jsonl
#> Loaded 9 queries with positives
#> Average positives per query: 1.56
#> Average negatives per query: 1.33
#> Converted triplets to qrels format: /Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/qrels.train.colbert.tsv
[Mar 06, 06:54:40] #> Loading qrels from /Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_verification/test/qrels.train.colbert.tsv ...
[Mar 06, 06:54:40] #> Loaded qrels for 9 unique queries with 1.56 positives per query on average.

[Mar 06, 06:54:49] #> Processing query 1 / 20

#> QueryTokenizer.tensorize(batch_text[

In [12]:
eval_res

defaultdict(dict,
            {'mrr': {10: 0.03958333333333333,
              100: 0.05317927170868347,
              1000: 0.05317927170868347},
             'success': {10: 0.2, 100: 0.45, 1000: 0.45},
             'recall': {10: 0.18333333333333332, 100: 0.45, 1000: 0.45},
             'precision': {30: 0.021666666666666667,
              50: 0.018421052631578942,
              100: 0.018421052631578942,
              200: 0.018421052631578942,
              500: 0.018421052631578942},
             'ndcg': {10: np.float64(0.07304411052376385),
              100: np.float64(0.14218625876618843)}})

In [23]:
queries=pd.read_csv('/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_curated/test/queries.train.colbert.tsv',sep='\t',header=None)
docs=pd.read_csv('/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_curated/test/corpus.train.colbert.tsv',sep='\t',header=None)
queries.columns=['qid','query']
docs.columns=['pid','doc']
import json

with open('/Users/pragalbh.devsingh/Documents/work/models/ColBERT/data/human_curated/test/triples.train.colbert.jsonl') as f:
    triples = [json.loads(line) for line in f]

pd.DataFrame(triples,columns=['qid','pid','nid'])

,qid,pid,nid
0,2,37,24
1,12,28,19
2,13,22,8
3,13,20,17
4,4,23,27
5,0,11,23
6,15,37,1
7,8,31,36
8,13,3,12
9,8,2,36


In [7]:
### calculating all posiives and negatives per query

(1/19+1/23 +1/7 +1/19 +1/13+1/3 +1/16 +1/3+1/5)/9



0.14418758946790983

In [8]:
(1/19 + 1/23 + 1/7 + 1/19 + 1/13 + 1/3 + 1/16 + 1/3 + 1/5) / 9

0.14418758946790983

In [24]:
docs['phrase_present']=docs['doc'].apply(lambda x: [phrase for phrase in all_phrases if phrase in x])

,pid,doc
0,0,Document 180: The service is poor. The quality...
1,1,Document 208: The price is average.
2,2,Document 184: The quality is poor.
3,3,Document 461: The service is average. The qual...
4,4,Document 71: The price is good.
5,5,Document 95: The quality is excellent. The ser...
6,6,Document 40: The price is excellent.
7,7,Document 4: The price is poor.
8,8,Document 85: The quality is poor. The price is...
9,9,Document 356: The quality is good.


In [35]:
queries['aspect']=queries['query'].apply(lambda x: x.split('||')[0])
queries['phrase']=queries['query'].apply(lambda x: x.split('||')[1].split(' ')[0])
all_aspects=queries['aspect'].unique()
all_phrases=queries['phrase'].unique()

In [36]:
all_phrases

array(['information', 'rating', 'review', 'how', 'tell', "what's",
       'describe', 'details'], dtype=object)

In [3]:
4/9

0.4444444444444444

In [5]:
(1/30+2/30+1/30+2/30+2/30+3/30+1/30+1/30+1/30)/9

0.05185185185185185

4/9

In [39]:
docs['phrase_present']=docs['doc'].apply(lambda x: [phrase for phrase in all_phrases if phrase.lower() in x.lower()])

In [41]:
docs.doc.iloc[0]

'Document 180: The service is poor. The quality is poor. '